In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DoubleType


In [0]:
dbutils.fs.ls('mnt/bronze')

[FileInfo(path='dbfs:/mnt/bronze/YTD 2018 _SWH.xlsx.csv', name='YTD 2018 _SWH.xlsx.csv', size=2080203, modificationTime=1710139690000),
 FileInfo(path='dbfs:/mnt/bronze/YTD 2018 _SWH.xlsx.parquet', name='YTD 2018 _SWH.xlsx.parquet', size=299230, modificationTime=1710161669000),
 FileInfo(path='dbfs:/mnt/bronze/recold0_2024-03-18 10:12:16.csv', name='recold0_2024-03-18 10:12:16.csv', size=1600301, modificationTime=1710756755000),
 FileInfo(path='dbfs:/mnt/bronze/recold1_2024-03-18 10:12:37.csv', name='recold1_2024-03-18 10:12:37.csv', size=1149625, modificationTime=1710756780000),
 FileInfo(path='dbfs:/mnt/bronze/recold2_2024-03-18 10:13:03.csv', name='recold2_2024-03-18 10:13:03.csv', size=1026247, modificationTime=1710756803000),
 FileInfo(path='dbfs:/mnt/bronze/recold3_2024-03-18 10:13:26.csv', name='recold3_2024-03-18 10:13:26.csv', size=891130, modificationTime=1710756825000)]

In [0]:
# Define schema for the DataFrame
schema = StructType([
    StructField("Sales_Region", StringType(), True),
    StructField("Plant_Id", StringType(), True),
    StructField("Sorg", StringType(), True),
    StructField("Dch", IntegerType(), True),
    StructField("Div", IntegerType(), True),
    StructField("Item_Categ", StringType(), True),
    StructField("Customer_Grp_Id", StringType(), True),
    StructField("Customer_Grp_Name", StringType(), True),
    StructField("Order_type", StringType(), True),
    StructField("Sales_Order", IntegerType(), True),
    StructField("Invoice_No", IntegerType(), True),
    StructField("Invoice_Date", DateType(), True),
    StructField("State_Name", StringType(), True),
    StructField("Sales_Zone", StringType(), True),
    StructField("Customer_Id", IntegerType(), True),
    StructField("Customer_Name", StringType(), True),
    StructField("Address", StringType(), True),
    StructField("Product_Id", IntegerType(), True),
    StructField("Product_Name", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Basic_Rate", DoubleType(), True),
    StructField("Special_Ra", DoubleType(), True),
    StructField("Basic_Amt", DoubleType(), True),
    StructField("Discount_A", IntegerType(), True),
    StructField("Net_Amt_DC", DoubleType(), True),
    StructField("D380", StringType(), True),
    StructField("Type_Calculations", StringType(), True),
    StructField("Product_Family", StringType(), True),
    StructField("Product_Type", StringType(), True),
    StructField("Product_Category", StringType(), True),
    StructField("Quantity_2", IntegerType(), True),
    StructField("Per_Coll", DoubleType(), True),
    StructField("Number_of_Coll", DoubleType(), True),
    StructField("Andris", StringType(), True),
    StructField("EWH_Groups", StringType(), True),
    StructField("Ecom", StringType(), True)
])

# Create DataFrame with the specified schema
result_df = spark.createDataFrame([], schema)

In [0]:
print(len(result_df.columns))

36


In [0]:
from datetime import datetime, timedelta
# Define a threshold for recent modification time (e.g., within the last 24 hours)
threshold_time = datetime.now() - timedelta(hours=24)
def clean_column_name(col_name):
        # Remove leading and trailing spaces
        cleaned_name = col_name.strip()
        # Replace spaces between words with underscores
        cleaned_name = re.sub(r'\s+', '_', cleaned_name)
        # Replace special characters with empty string
        cleaned_name = re.sub(r'[^a-zA-Z0-9_]', '', cleaned_name)
        return cleaned_name

def NullValueReplace(df):
    # Iterate over each column
    for column_name in df.columns:
        # Check if the column contains null values
        if df.filter(df[column_name].isNull()).count() > 0:
            # Get the data type of the column
            column_type = df.schema[column_name].dataType
        # Replace null values based on the data type
        if column_type in (IntegerType(), LongType(), ShortType(), ByteType()):
            df = df.withColumn(column_name, when(df[column_name].isNull(), 0).otherwise(df[column_name]))
        elif column_type == StringType():
            df = df.withColumn(column_name, when(df[column_name].isNull(), "Other").otherwise(df[column_name]))
        elif column_type == DoubleType():
            df = df.withColumn(column_name, when(df[column_name].isNull(), 0.0).otherwise(df[column_name]))
    return df

def DropUnwantedColumns(df):
    df=df.drop('Product_Fa','Material_Description_2','EWH_Region_Classification','ES_State',
            'State_with_Haryana_1_2','Solar_Group','For_Split','Commercial_vs_Domestic')
        #Dropping Additional column which is present only in first sheet
    for column_name in df.columns:
        if column_name=='Source':
            df=df.drop('Source')
        if column_name=='PF01':
            df=df.drop('PF01')
        if column_name=='Weeks':
            df=df.drop('Weeks')
    return df


In [0]:
for filePath in dbutils.fs.ls('mnt/bronze'):
    if datetime.fromtimestamp(filePath.modificationTime / 1000) > threshold_time:
        path=filePath[0]
        print(path)
        df=spark.read.format("csv").options(header=True,inferschema=True).load(path)
        # Get the current column names
        current_columns = df.columns
        # Clean column names
        cleaned_columns = [clean_column_name(col_name) for col_name in current_columns]
        # Rename columns
        df = df.toDF(*cleaned_columns)
        #Dropping Unwnated columns
        df=DropUnwantedColumns(df)
        #Replacing Null values 
        df=NullValueReplace(df)
        #Schema Standerdization
        df=df.withColumnRenamed('REF','Sales_Region')\
        .withColumnRenamed('Sales_Orde','Sales_Order')\
        .withColumnRenamed('Invoice_Da','Invoice_Date')\
        .withColumnRenamed('Plant','Plant_Id')\
        .withColumnRenamed('Cust_Grp','Customer_Grp_Id')\
        .withColumnRenamed('Customer_G','Customer_Grp_Name')\
        .withColumnRenamed('Sold_To','Customer_Id')\
        .withColumnRenamed('Sold_to_Party_Name','Customer_Name')\
        .withColumnRenamed('Sold_To_City','Address')\
        .withColumnRenamed('Material','Product_Id')\
        .withColumnRenamed('Material_Description','Product_Name')\
        .withColumnRenamed('Type','Product_Type')\
        .withColumnRenamed('Solar_Type','Product_Category')
        
        # # df.printSchema()
        print(len(df.columns))
        
        # if len(df.columns)==36:
        #     df.printSchema()
        #     break
        result_df=result_df.unionAll(df)

dbfs:/mnt/bronze/recold0_2024-03-18 10:12:16.csv
47
dbfs:/mnt/bronze/recold1_2024-03-18 10:12:37.csv
44
dbfs:/mnt/bronze/recold2_2024-03-18 10:13:03.csv
44
dbfs:/mnt/bronze/recold3_2024-03-18 10:13:26.csv
44


In [0]:
result_df.count()

10354

In [0]:
df_single_partition = result_df.repartition(1)

In [0]:
opt_path='mnt/silver'
# result_df.write.option("header","true").csv(opt_path+'/'+'Df.csv',mode='ignore')
df_single_partition.write.parquet(opt_path,mode='overwrite')

Code end here

In [0]:
# from datetime import datetime, timedelta
# # Define a threshold for recent modification time (e.g., within the last 24 hours)
# threshold_time = datetime.now() - timedelta(hours=48)

# # Filter the list to select only the recently added files
# recent_files = list([file_info for file_info in dbutils.fs.ls('mnt/bronze') if datetime.fromtimestamp(file_info.modificationTime / 1000) > threshold_time])

# # Print the list of recently added files
# for file_path in recent_files:
#     print(file_path) 

FileInfo(path='dbfs:/mnt/bronze/datasheet_0.csv', name='datasheet_0.csv', size=1599840, modificationTime=1710326743000)
FileInfo(path='dbfs:/mnt/bronze/datasheet_1.csv', name='datasheet_1.csv', size=1149194, modificationTime=1710326769000)
FileInfo(path='dbfs:/mnt/bronze/datasheet_2.csv', name='datasheet_2.csv', size=1025816, modificationTime=1710326787000)
FileInfo(path='dbfs:/mnt/bronze/datasheet_3.csv', name='datasheet_3.csv', size=890699, modificationTime=1710326807000)


In [0]:
# path1=recent_files[0][0]
# print(path1)

dbfs:/mnt/bronze/datasheet_0.csv


In [0]:
# df=spark.read.format("csv").options(header=True,inferschema=True).load(path1)
# df.printSchema()

root
 |-- #REF!: string (nullable = true)
 |-- Plant: string (nullable = true)
 |-- Sorg: string (nullable = true)
 |-- Dch: integer (nullable = true)
 |-- Div: integer (nullable = true)
 |-- Item Categ: string (nullable = true)
 |-- Cust Grp: string (nullable = true)
 |-- Customer G: string (nullable = true)
 |-- Order type: string (nullable = true)
 |-- Sales Orde: integer (nullable = true)
 |-- Invoice No: integer (nullable = true)
 |-- Invoice Da: date (nullable = true)
 |-- Weeks: string (nullable = true)
 |-- State Name: string (nullable = true)
 |-- Sales Zone: string (nullable = true)
 |-- Sold_To: integer (nullable = true)
 |-- Sold_to Party Name: string (nullable = true)
 |-- Sold_To City: string (nullable = true)
 |-- Product Fa: string (nullable = true)
 |-- Material: integer (nullable = true)
 |-- Material Description: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- PF01: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |--  Basic Rate

In [0]:
# print(len(df.columns))

47


column name standerdization 

In [0]:
# def clean_column_name(col_name):
#     # Remove leading and trailing spaces
#     cleaned_name = col_name.strip()
#     # Replace spaces between words with underscores
#     cleaned_name = re.sub(r'\s+', '_', cleaned_name)
#     # Replace special characters with empty string
#     cleaned_name = re.sub(r'[^a-zA-Z0-9_]', '', cleaned_name)
#     return cleaned_name

# # Get the current column names
# current_columns = df.columns

# # Clean column names
# cleaned_columns = [clean_column_name(col_name) for col_name in current_columns]

# # Rename columns
# df = df.toDF(*cleaned_columns)
# # print(cleaned_columns)

In [0]:
# df.printSchema()

root
 |-- REF: string (nullable = true)
 |-- Plant: string (nullable = true)
 |-- Sorg: string (nullable = true)
 |-- Dch: integer (nullable = true)
 |-- Div: integer (nullable = true)
 |-- Item_Categ: string (nullable = true)
 |-- Cust_Grp: string (nullable = true)
 |-- Customer_G: string (nullable = true)
 |-- Order_type: string (nullable = true)
 |-- Sales_Orde: integer (nullable = true)
 |-- Invoice_No: integer (nullable = true)
 |-- Invoice_Da: date (nullable = true)
 |-- Weeks: string (nullable = true)
 |-- State_Name: string (nullable = true)
 |-- Sales_Zone: string (nullable = true)
 |-- Sold_To: integer (nullable = true)
 |-- Sold_to_Party_Name: string (nullable = true)
 |-- Sold_To_City: string (nullable = true)
 |-- Product_Fa: string (nullable = true)
 |-- Material: integer (nullable = true)
 |-- Material_Description: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- PF01: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Basic_Rate: d

Dropping Unwanted Columns

In [0]:
# df=df.drop('Product_Fa','Material_Description_2','EWH_Region_Classification','ES_State',
#             'State_with_Haryana_1_2','Solar_Group','For_Split','Commercial_vs_Domestic')

# #Dropping Additional column which is present only in first sheet
# for column_name in df.columns:
#     if column_name=='Source':
#         df=df.drop('Source')
#     if column_name=='PF01':
#         df=df.drop('PF01')


In [0]:
# print(len(df.columns))

37


In [0]:
# df.printSchema()

root
 |-- REF: string (nullable = true)
 |-- Plant: string (nullable = true)
 |-- Sorg: string (nullable = true)
 |-- Dch: integer (nullable = true)
 |-- Div: integer (nullable = true)
 |-- Item_Categ: string (nullable = true)
 |-- Cust_Grp: string (nullable = true)
 |-- Customer_G: string (nullable = true)
 |-- Order_type: string (nullable = true)
 |-- Sales_Orde: integer (nullable = true)
 |-- Invoice_No: integer (nullable = true)
 |-- Invoice_Da: date (nullable = true)
 |-- Weeks: string (nullable = true)
 |-- State_Name: string (nullable = true)
 |-- Sales_Zone: string (nullable = true)
 |-- Sold_To: integer (nullable = true)
 |-- Sold_to_Party_Name: string (nullable = true)
 |-- Sold_To_City: string (nullable = true)
 |-- Material: integer (nullable = true)
 |-- Material_Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Basic_Rate: double (nullable = true)
 |-- Special_Ra: double (nullable = true)
 |-- Basic_Amt: double (nullable = true)
 |-- Disc

Checking null values and replacing it as per data type


In [0]:
# # Iterate over each column
# for column_name in df.columns:
#     # Check if the column contains null values
#     if df.filter(df[column_name].isNull()).count() > 0:
#         # Get the data type of the column
#         column_type = df.schema[column_name].dataType
#         # Replace null values based on the data type
#         if column_type in (IntegerType(), LongType(), ShortType(), ByteType()):
#             df = df.withColumn(column_name, when(df[column_name].isNull(), 0).otherwise(df[column_name]))
#         elif column_type == StringType():
#             df = df.withColumn(column_name, when(df[column_name].isNull(), "Other").otherwise(df[column_name]))
#         elif column_type == DoubleType():
#             df = df.withColumn(column_name, when(df[column_name].isNull(), 0.0).otherwise(df[column_name]))

column Rename 
 

In [0]:
# clm=df.columns
# print(clm)

['REF', 'Plant', 'Sorg', 'Dch', 'Div', 'Item_Categ', 'Cust_Grp', 'Customer_G', 'Order_type', 'Sales_Orde', 'Invoice_No', 'Invoice_Da', 'Weeks', 'State_Name', 'Sales_Zone', 'Sold_To', 'Sold_to_Party_Name', 'Sold_To_City', 'Material', 'Material_Description', 'Quantity', 'Basic_Rate', 'Special_Ra', 'Basic_Amt', 'Discount_A', 'Net_Amt_DC', 'D380', 'Type_Calculations', 'Product_Family', 'Type', 'Solar_Type', 'Quantity_2', 'Per_Coll', 'Number_of_Coll', 'Andris', 'EWH_Groups', 'Ecom']


In [0]:
# df=df.withColumnRenamed('REF','Sales_Region')\
#     .withColumnRenamed('Sales_Orde','Sales_Order')\
#     .withColumnRenamed('Invoice_Da','Invoice_Date')\
#     .withColumnRenamed('Plant','Plant_Id')\
#     .withColumnRenamed('Cust_Grp','Customer_Grp_Id')\
#     .withColumnRenamed('Customer_G','Customer_Grp_Name')\
#     .withColumnRenamed('Sold_To','Customer_Id')\
#     .withColumnRenamed('Sold_to_Party_Name','Customer_Name')\
#     .withColumnRenamed('Sold_To_City','Address')\
#     .withColumnRenamed('Material','Product_Id')\
#     .withColumnRenamed('Material_Description','Product_Name')\
#     .withColumnRenamed('Type','Product_Type')\
#     .withColumnRenamed('Solar_Type','Product_Category')

In [0]:
# print(df.columns)

['Sales_Region', 'Plant_Id', 'Sorg', 'Dch', 'Div', 'Item_Categ', 'Customer_Grp_Id', 'Customer_Grp_Name', 'Order_type', 'Sales_Order', 'Invoice_No', 'Invoice_Date', 'Weeks', 'State_Name', 'Sales_Zone', 'Customer_Id', 'Customer_Name', 'Address', 'Product_Id', 'Product_Name', 'Quantity', 'Basic_Rate', 'Special_Ra', 'Basic_Amt', 'Discount_A', 'Net_Amt_DC', 'D380', 'Type_Calculations', 'Product_Family', 'Product_Type', 'Product_Category', 'Quantity_2', 'Per_Coll', 'Number_of_Coll', 'Andris', 'EWH_Groups', 'Ecom']


In [0]:
# clm= ['Sales_Region', 'Plant_Id', 'Sorg', 'Dch', 'Div', 'Item_Categ', 'Customer_Grp_Id', 'Customer_Grp_Name', 'Order_type', 'Sales_Order', 'Invoice_No', 'Invoice_Date', 'Weeks', 'State_Name', 'Sales_Zone', 'Customer_Id', 'Customer_Name', 'Address', 'Product_Id', 'Product_Name', 'Quantity', 'Basic_Rate', 'Special_Ra', 'Basic_Amt', 'Discount_A', 'Net_Amt_DC', 'D380', 'Type_Calculations', 'Product_Family', 'Product_Type', 'Product_Category', 'Quantity_2', 'Per_Coll', 'Number_of_Coll', 'Andris', 'EWH_Groups', 'Ecom']
# print(len(clm))

37


Before it merge all sheet data to single df 

Creating fact and dimentions

In [0]:
# def cust_dim(df):
#   customer_df=df.select('Customer_Id','Customer_Name','Address')
#   df=df.drop('Customer_Name','Address') #dropping columns from main table which are taken in cusomer df
#   customer_df=customer_df.dropDuplicates(['Customer_Id'])
#   # customer_df.show()
#   return customer_df, df

# def plant_dim(df):
#   plant_df=df.select('Plant_Id','Sorg','Dch','Div')
#   #dropping columns from main table which are taken in pant df
#   df=df.drop('Sorg','Dch','Div')
#   plant_df=plant_df.dropDuplicates(['Plant_Id','Dch'])
#   return plant_df, df

# def custgrp_dim(df):
#   custgrp_df=df.select('Customer_Grp_Id','Customer_Grp_Name')
#   print("Before",custgrp_df.count())
#   #dropping columns from main table which are taken in pant df
#   df=df.drop('Customer_Grp_Name')
#   custgrp_df=custgrp_df.dropDuplicates(['Customer_Grp_Id'])
#   print("After",custgrp_df.count())
#   return custgrp_df, df

# def product_dim(df):
#   material_df=df.select('Product_Id','Product_Name','Product_Category','Product_Type','Andris','EWH_Groups')
#   print("Before",material_df.count())
#   #dropping columns from main table which are taken in pant df
#   df=df.drop('Product_Name','Product_Category','Product_Type','Andris','EWH_Groups')
#   material_df=material_df.dropDuplicates(['Product_Id'])
#   print("After",material_df.count())
#   return material_df, df

# def location_dim(df):
#   location_df=df.select(col('State_Name').alias('StateName'),col('Sales_Zone').alias('SalesZone'),col('Sales_Region').alias('SalesRegion'))
#   location_df=location_df.dropDuplicates(['StateName','SalesZone','SalesRegion'])
#   # Add a unique id column to the existing dataframe
#   location_df = location_df.withColumn("StateId", monotonically_increasing_id())
#   # Convert the id column to a string
#   location_df = location_df.withColumn("StateId", concat(lit("S"), col("StateId").cast("string")))
#   return location_df, df

Creating Fact and Dimentions

In [0]:
# #Creating Customer Dimention
# customer_df,df=cust_dim(df)
# #Creating Plant Dimention
# plant_df,df=plant_dim(df)
# #Creating Customer_group Dimention
# custgrp_df,df=custgrp_dim(df)
# #Creating product Dimention
# product_df,df=product_dim(df)
# #Creating Location Dimention
# location_df,df=location_dim(df)
# location_df.dropDuplicates(['StateId'])
# #addting loaction_id to fact data frame 
# df = df.join(location_df,
#                     (df.State_Name == location_df.StateName) &
#                     (df.Sales_Zone == location_df.SalesZone) &
#                     (df.Sales_Region == location_df.SalesRegion),
#                     "left")
# df=df.drop("State_Name","Sales_Zone","Sales_Region","StateName","SalesZone","SalesRegion")
# #Df is fact data frame

+-----------------+---------+-----------+-------+
|        StateName|SalesZone|SalesRegion|StateId|
+-----------------+---------+-----------+-------+
|            Bihar|    INEST|      INEST|     S0|
|      Pondicherry|    INSR2|      INSRG|     S1|
|            Delhi|    INCSD|      INCSD|     S2|
|              Goa|    INWST|      INWST|     S3|
|    Uttar Pradesh|    INNRG|      INNRG|     S4|
|           Punjab|    INNRG|      INNRG|     S5|
|       Tamil Nadu|    INSR1|      INSRG|     S6|
|      Maharashtra|    INWST|      INWST|     S7|
|      West Bengal|    INEST|      INEST|     S8|
|       Tamil Nadu|    INCSD|      INCSD|     S9|
|Jammu and Kashmir|    INCSD|      INCSD|    S10|
|   Madhya Pradesh|    INCSD|      INCSD|    S11|
|           Orissa|    INEST|      INEST|    S12|
|          Gujarat|    INWST|      INWST|    S13|
|           Kerala|    INSRG|      INSRG|    S14|
|           Kerala|    INCSD|      INCSD|    S15|
|           Kerala|    INSR1|      INSRG|    S16|


In [0]:
# demo_df.write.option("header","true").csv(opt_path+'/'+'demodf',mode='ignore')
# location_df.write.option("header","true").csv(opt_path+'/'+'locationDf',mode='ignore')

In [0]:

# df.write.option("header","true").csv(opt_path+'/'+'salesfactData',mode='ignore')